In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt

In [4]:
df = pd.read_csv('../data/processed/india_aq_clean.csv' , parse_dates = ['datetime_local'])

In [7]:
pm25 = df[df['parameter'] == 'pm25'].copy()

pm25.groupby('location')['value'].count().sort_values(ascending = False)

location
Punjabi Bagh Delhi      2000
Zoo Park Hyderabad      2000
R K Puram Delhi         1997
Anand Vihar Delhi       1981
Vikas Sadan Gurugram    1948
Lalbagh Mumbai          1000
Collectorate Gaya        999
Alandur Chennai          989
Income Tax Delhi         876
Name: value, dtype: int64

##### Filter to one station, set datetime index, resample to daily

In [9]:
station = pm25[pm25['location'] == 'Punjabi Bagh Delhi'].copy()
station = station.set_index('datetime_local')
daily = station['value'].resample('D').mean().dropna().to_frame()

# to_frame() converts Series → DataFrame (needed for adding columns later)
print(f"Days of data: {len(daily)}")
print(daily.head(10))

Days of data: 28
                                value
datetime_local                       
2016-02-05 00:00:00+05:30  257.333333
2016-02-06 00:00:00+05:30  243.605634
2016-02-07 00:00:00+05:30  189.089552
2016-02-08 00:00:00+05:30  146.400000
2016-02-09 00:00:00+05:30  203.206349
2016-02-10 00:00:00+05:30  265.353846
2016-02-11 00:00:00+05:30  132.405797
2016-02-12 00:00:00+05:30  147.900000
2016-02-13 00:00:00+05:30  109.250000
2016-02-14 00:00:00+05:30  119.622951


##### Time features

In [10]:
daily['month'] = daily.index.month
daily['day_of_week'] = daily.index.dayofweek
daily['is_weekend'] = (daily['day_of_week'] >= 5).astype(int)

print(daily[["value", "month", "day_of_week", "is_weekend"]].head(10))

                                value  month  day_of_week  is_weekend
datetime_local                                                       
2016-02-05 00:00:00+05:30  257.333333      2            4           0
2016-02-06 00:00:00+05:30  243.605634      2            5           1
2016-02-07 00:00:00+05:30  189.089552      2            6           1
2016-02-08 00:00:00+05:30  146.400000      2            0           0
2016-02-09 00:00:00+05:30  203.206349      2            1           0
2016-02-10 00:00:00+05:30  265.353846      2            2           0
2016-02-11 00:00:00+05:30  132.405797      2            3           0
2016-02-12 00:00:00+05:30  147.900000      2            4           0
2016-02-13 00:00:00+05:30  109.250000      2            5           1
2016-02-14 00:00:00+05:30  119.622951      2            6           1


##### Lag features

In [11]:
daily['lag_1'] = daily['value'].shift(1)
daily['lag_2'] = daily['value'].shift(2)
daily['lag_3'] = daily['value'].shift(3)

print(daily[["value", "lag_1", "lag_2", "lag_3"]].head(5))

                                value       lag_1       lag_2       lag_3
datetime_local                                                           
2016-02-05 00:00:00+05:30  257.333333         NaN         NaN         NaN
2016-02-06 00:00:00+05:30  243.605634  257.333333         NaN         NaN
2016-02-07 00:00:00+05:30  189.089552  243.605634  257.333333         NaN
2016-02-08 00:00:00+05:30  146.400000  189.089552  243.605634  257.333333
2016-02-09 00:00:00+05:30  203.206349  146.400000  189.089552  243.605634


##### Rolling features

In [12]:
daily['roll_3_mean'] = daily['value'].rolling(window = 3).mean()
daily['roll_7_mean'] = daily['value'].rolling(window = 7).mean()
daily['roll_3_std'] = daily['value'].rolling(window = 3).std()

print(daily[["value", "roll_3_mean", "roll_7_mean", "roll_3_std"]].head(10))

                                value  roll_3_mean  roll_7_mean  roll_3_std
datetime_local                                                             
2016-02-05 00:00:00+05:30  257.333333          NaN          NaN         NaN
2016-02-06 00:00:00+05:30  243.605634          NaN          NaN         NaN
2016-02-07 00:00:00+05:30  189.089552   230.009506          NaN   36.096321
2016-02-08 00:00:00+05:30  146.400000   193.031729          NaN   48.722576
2016-02-09 00:00:00+05:30  203.206349   179.565300          NaN   29.576576
2016-02-10 00:00:00+05:30  265.353846   204.986732          NaN   59.496905
2016-02-11 00:00:00+05:30  132.405797   200.321997   205.342073   66.520941
2016-02-12 00:00:00+05:30  147.900000   181.886548   189.708740   72.698762
2016-02-13 00:00:00+05:30  109.250000   129.851932   170.515078   19.451151
2016-02-14 00:00:00+05:30  119.622951   125.590984   160.591278   20.004217


##### DRopping Rows with Any nan

In [13]:
daily_clean = daily.dropna()
print(f"Before dropping Nan {len(daily)} rows")
print(f"After dropping Nan {len(daily_clean)} rows")
print(f"Lost:   {len(daily) - len(daily_clean)} rows (to lag/rolling warmup)")

print(daily_clean.head())

Before dropping Nan 28 rows
After dropping Nan 22 rows
Lost:   6 rows (to lag/rolling warmup)
                                value  month  day_of_week  is_weekend  \
datetime_local                                                          
2016-02-11 00:00:00+05:30  132.405797      2            3           0   
2016-02-12 00:00:00+05:30  147.900000      2            4           0   
2016-02-13 00:00:00+05:30  109.250000      2            5           1   
2016-02-14 00:00:00+05:30  119.622951      2            6           1   
2016-02-15 00:00:00+05:30  117.420290      2            0           0   

                                lag_1       lag_2       lag_3  roll_3_mean  \
datetime_local                                                               
2016-02-11 00:00:00+05:30  265.353846  203.206349  146.400000   200.321997   
2016-02-12 00:00:00+05:30  132.405797  265.353846  203.206349   181.886548   
2016-02-13 00:00:00+05:30  147.900000  132.405797  265.353846   129.851932   
2016

##### Define features (X) and target (y)

In [15]:
feature_cols = ['month', 'day_of_week', 'is_weekend', 'lag_1', 'lag_2', 'lag_3', 'roll_3_mean', 'roll_7_mean', 'roll_3_std']

x = daily_clean[feature_cols]
y = daily_clean['value']

print(f"Features shape: {x.shape}")  # (22, 9) → 22 rows, 9 features
print(f"Target shape:   {y.shape}")  # (22,) → 22 values to predict
print(f"\nFeature columns:\n{list(x.columns)}")

Features shape: (22, 9)
Target shape:   (22,)

Feature columns:
['month', 'day_of_week', 'is_weekend', 'lag_1', 'lag_2', 'lag_3', 'roll_3_mean', 'roll_7_mean', 'roll_3_std']
